# 🚀 Advanced PySpark Performance Tuning & Optimization (Part 2)
This notebook covers the remaining advanced optimization techniques for production PySpark workloads: Manual Salting for Data Skew, Memory Persistence & Storage Levels, Window Function Framing Optimization, and High-Performance Complex Schema Flattening.

--- 
## 6. Salting for Severe Data Skew Mitigation
### **Concept & Explanation**
* **The Bottleneck:** When a join key is heavily skewed (e.g., 80% of rows have `customer_id = 0`), standard shuffles group all matching keys onto a single executor task, causing severe OOM errors or long tail task latency.
* **How Salting Works:** We append a random integer ('salt') between `0` and `N-1` to the skewed key in the large table, while replicating the lookup rows in the secondary table across all `N` salt values. This breaks the single hot key into `N` parallel sub-keys across different partitions.
* **Production Best Practice:** Use manual salting when AQE skew join is insufficient (e.g., extremely complex multi-key joins or when running on older Spark engines without AQE).

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Optimization_6_Salting_Technique") \
    .master("local[*]") \
    .getOrCreate()

# Step 1: Create heavily skewed dataset
num_rows = 500000
large_df = spark.range(0, num_rows).withColumn(
    "skewed_key", F.when(F.rand() < 0.8, 0).otherwise((F.rand() * 50).cast("int"))
)
lookup_df = spark.range(0, 50).withColumnRenamed("id", "skewed_key").withColumn("metadata", F.lit("Category_Info"))

# Step 2: Apply Salting (Salt factor N = 4)
SALT_FACTOR = 4

# Add salt (0 to N-1) to large table
salted_large = large_df.withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
                        .withColumn("salted_key", F.concat_ws("_", F.col("skewed_key"), F.col("salt")))

# Replicate lookup table rows across all N salts
salt_array = F.array([F.lit(i) for i in range(SALT_FACTOR)])
salted_lookup = lookup_df.withColumn("salt", F.explode(salt_array)) \
                         .withColumn("salted_key", F.concat_ws("_", F.col("skewed_key"), F.col("salt")))

# Step 3: Join on salted_key (Distributes hot key across N partitions)
salted_joined = salted_large.join(salted_lookup, "salted_key")
print(f"Salted Join Result Count: {salted_joined.count():,}")
salted_joined.explain(True)

--- 
## 7. Efficient Memory Persistence & Storage Levels
### **Concept & Explanation**
* **Storage Levels Compared:**
  * `MEMORY_ONLY`: Deserialized Java objects in JVM memory. Fast, but high GC pressure.
  * `MEMORY_AND_DISK_SER`: Serialized byte arrays in memory with disk spilling fallback. Highly compact, reduces Garbage Collection (GC) overhead.
* **Production Best Practice:** Avoid default `.cache()` (which uses `MEMORY_AND_DISK` deserialized) on multi-gigabyte DataFrames. Always use `persist(StorageLevel.MEMORY_AND_DISK_SER)` for large intermediate tables and remember to call `.unpersist()` as soon as the dataset is no longer needed.

In [ ]:
from pyspark.storagelevel import StorageLevel
import time

# Step 1: Generate synthetic DataFrame
df_persist = spark.range(0, 2000000).withColumn("v1", F.rand()).withColumn("v2", F.concat_ws("_", F.lit("val"), F.col("id")))

# Step 2: Persist with MEMORY_AND_DISK_SER (Compact Byte Array Storage)
print("=== Persisting DataFrame with MEMORY_AND_DISK_SER ===")
df_persist.persist(StorageLevel.MEMORY_AND_DISK_SER)

start = time.time()
count_1 = df_persist.count() # Populates storage cache
print(f"First action (Cache Miss + Populate): {time.time() - start:.2f}s | Rows: {count_1:,}")

start = time.time()
count_2 = df_persist.count() # Reads from storage cache
print(f"Second action (Cache Hit): {time.time() - start:.2f}s | Rows: {count_2:,}")

# Clean up memory storage
df_persist.unpersist()
print("DataFrame successfully unpersisted from storage.")

--- 
## 8. Window Function Frame Optimization (`rowsBetween` vs `rangeBetween`)
### **Concept & Explanation**
* **The Danger of Default Framing:** When using `.orderBy()` inside a Window specification without an explicit frame, PySpark defaults to `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Range framing requires Spark to evaluate ties and values dynamically, causing huge memory buffer overhead.
* **The Fix (`rowsBetween`):** Explicitly specifying `Window.rowsBetween(Window.unboundedPreceding, Window.currentRow)` tells Spark to operate on exact physical row offsets, allowing WholeStageCodegen to run in a continuous streaming pass.
* **Production Best Practice:** Always specify explicit row boundaries for cumulative window aggregates.

In [ ]:
from pyspark.sql.window import Window

# Step 1: Create transaction dataset
window_data = [(f"user_{i % 1000}", i, float(i * 10)) for i in range(100000)]
df_win = spark.createDataFrame(window_data, ["user_id", "timestamp", "amount"])

# Step 2: Optimized Window Specification using explicit physical row framing
opt_window = Window.partitionBy("user_id") \
                   .orderBy("timestamp") \
                   .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_cum_sum = df_win.withColumn("running_total", F.sum("amount").over(opt_window))
print("=== Physical Plan: Physical Row Framing Optimization ===")
df_cum_sum.explain(True)

--- 
## 9. High-Performance Complex Schema Flattening (Avoiding Multiple Explosions)
### **Concept & Explanation**
* **The Bottleneck:** Multiple sequential `.select()` and `F.explode()` operations trigger costly re-evaluations and intermediate row projections.
* **The Fix (`F.inline` / Higher-Order Functions):** Using native higher-order functions like `F.transform()`, `F.filter()`, or `F.inline()` flattens arrays of structs in a single execution step without creating intermediate temporary data structures.

In [ ]:
# Step 1: Create DataFrame with array of structs
nested_data = [
    (1, [{"item": "A", "price": 10}, {"item": "B", "price": 20}]),
    (2, [{"item": "C", "price": 15}, {"item": "D", "price": 25}])
]
df_nested = spark.createDataFrame(nested_data, ["order_id", "items"])

# Step 2: High-Performance Flattening using inline
print("=== Efficient Array Struct Expansion via F.inline ===")
df_flat = df_nested.select("order_id", F.inline("items"))
df_flat.show()
df_flat.explain(True)

spark.stop()